[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/modal_xpbd/modal_xpbd.ipynb)

# Flexible Bodies, Rigidly Joined

Each body is a girder of bars. Its rigid motion is a motor, and its deformation is a combination of its lowest vibration modes. Bodies are connected by point constraints between their corners. Two cases are simulated. A chain of girders, one constraint per joint, swings under gravity. A beam of girders, two constraints per joint and fixed at both ends, is compressed until it buckles. Time stepping is extended position-based dynamics, with all constraints solved together in one sparse solve per step. The couplings between bodies and constraints are sparse extensors.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from dataclasses import replace

import numpy as np
from IPython.display import Image, display

from numga import stack
from numga.sparse import SparseExtensor, spdiag
from examples.animation import save_animation
from examples.mechanics import lie_integrators as lie
from examples.mechanics.modal_xpbd import core, render
from examples.mechanics.modal_xpbd.core import (
    Bodies, Direction, Force, Forque, Motor, Constraints, Point, Scalar, Shape, Twist, mv, truss,
)
from examples.mechanics.modal_xpbd.scenarios import clamped_beam, critical, displaced, hinged_chain

# The chain's girder: its cells, length and height, its bars' stiffness, its density, and the
# vibration modes it keeps.
CELLS, LENGTH, HEIGHT = 4, 1.0, 0.2
STIFFNESS, DENSITY, MODES = 300.0, 1.0, 8
# The chain's moving girders, their modes' damping ratio, and gravity.
LINKS = 4
DAMPING = np.array([0.02])
GRAVITY = (mv.y * -4).dual()                                                 # [] Direction
# The swing's time step, its frames and the steps between them, and the frame duration.
INTERVAL, FRAMES, SUBSTEPS = 0.002, 200, 15
DURATION_MS = 30
# The modes drawn, and how far each is drawn displaced.
SHOWN, DRAWN = 4, 0.03
# The beam: its moving girders, their cells of unit length, their height, the bars' stiffness, the
# modes' damping ratio, the final end displacement, its frames and the steps between them.
BEAM_GIRDERS, BEAM_CELLS, BEAM_HEIGHT = 8, 32, 1.0
BEAM_STIFFNESS, BEAM_DAMPING = 1e11, np.array([3.0])
END_DISPLACEMENT, BEAM_FRAMES, BEAM_SUBSTEPS, BEAM_INTERVAL = 0.15, 120, 20, 0.002
NO_GRAVITY = (mv.y * 0).dual()                                               # [] Direction

## 1. The girder and its vibration modes

`truss` lays out the girder's points and bars. Displacements and forces are represented by the lines `x` and `y` through the origin, `Force`, the polars of the directions; `force & direction` is the work of the force along the direction. `ends` is a sparse extensor from fields on the points to fields on the bars, with cells minus one and one, so `ends * positions` is the difference vector of every bar. Each bar is the map `direction * (direction | Force)`, `Force <- Force`, scaled by its stiffness over its length. Map cells compose by call, so the stiffness is `~ends * bars(ends * Force)`. Its eigenfields against the lumped masses are the vibration modes. The three modes at zero are rigid motions, carried by the motor instead. The plot shows the four lowest.

In [ ]:
positions, edges = truss(CELLS, LENGTH, HEIGHT)                              # [vertices] Force, [edges, ends]
vertices = positions.shape[0]
# Each bar's tail taken from its head.
ends = SparseExtensor.from_columns(edges, mv.scalar((np.ones_like(edges) * [-1, 1])[..., None]), vertices)  # [edges, vertices] Scalar
difference = ends * positions                                                # [edges] Force
lengths = difference.norm()                                                  # [edges] Scalar
directions = difference / lengths                                            # [edges] Force
# Each bar measures extension along itself and pulls back along the same direction.
bars = spdiag(directions * (directions | Force) * STIFFNESS / lengths)       # [edges, edges] Force <- Force
masses = np.full(vertices, DENSITY * LENGTH * HEIGHT / vertices)
weights = spdiag(mv.scalar(masses[:, None])) * Force                         # [vertices, vertices] Force <- Force
# The stiffness against the masses; its three motions at zero are rigid and belong to the motor.
values, fields = (~ends * bars(ends * Force)).eigh(weights, 3 + MODES)       # [3 + modes] Scalar, [3 + modes, vertices] Force
values, fields = values[3:], fields[3:]                                      # [modes] Scalar, [modes, vertices] Force
centre = (positions * masses).sum(axis=0) / masses.sum()                     # [] Force
rest = (positions - centre + mv.w).dual()                                    # [vertices] Point
inertia = ((rest & rest.commutator(Twist)) * masses).sum(axis=0)             # [] Forque <- Twist
shape = Shape(rest, fields.dual(), values.square_root(), 1 / values, masses, edges, inertia)

print(ends.gatype, ends.shape)
print(bars.gatype, bars.shape)
render.mode_shapes(shape.rest, shape.rest + shape.modes[:SHOWN] * DRAWN, shape.edges);

## 2. Bodies and constraints

A body's points are its rest points plus its modes weighted by their amplitudes, transformed by its motor, `motor >> local_points`. Body state is batched, one field per quantity with an axis over the bodies. A constraint holds an anchor point on one body to an anchor point on another. A hinge is one constraint, at the upper corners of two adjacent girders, and a splice is two, one at each corner. In the chain, the first girder is fixed by a zero inverse inertia.

In [ ]:
def points(bodies: Bodies, shape: Shape) -> Point:
    """The bodies' points in the world."""
    # Each body's rest points moved by its modes, in its own frame.
    local_points = shape.rest + (shape.modes[:, None] * bodies.amplitudes[..., None]).sum(axis=-3)  # [..., bodies, vertices] Point
    return bodies.motor[..., None] >> local_points                         # [..., bodies, vertices] Point


bodies, constraints = hinged_chain(shape, LINKS, DAMPING)
render.draw_chain(points(bodies, shape), shape.edges);

## 3. Constraints as sparse extensors

For an anchor, `anchor.commutator(Twist)` is the map from a twist of its body to the anchor's displacement, `Direction <- Twist`, and the motor transforms it to the world frame. The difference of these maps between the two anchors of each constraint, over all constraints and bodies, is the sparse extensor `rigid`, `[constraints, bodies] Direction <- Twist`, with cells only at each constraint's two bodies. Its adjugate, `[bodies, constraints] Forque <- Force`, maps forces at the constraints to forques on the bodies, and `rigid.adjugate()(forces) & twists == forces & rigid(twists)`, summed over the elements. `modal` maps the bodies' mode amplitudes to the gaps in the same way, one map per mode along a leading axis, and `modal_load` maps forces at the constraints to loads on the modes, one map per mode.

In [ ]:
def coupling(bodies: Bodies, constraints: Constraints) -> tuple[SparseExtensor, SparseExtensor, SparseExtensor, Direction]:
    """The sparse maps from the bodies' rigid motion and modes to the constraint gaps, and the gaps."""
    body_count, constraint_count = bodies.motor.shape[-1], len(constraints.body_idx)
    # The motors of the two bodies each constraint joins.
    motor = bodies.motor[..., constraints.body_idx]                        # [..., constraints, sides] Motor
    # The constraint's anchor on each of the two bodies, in that body's frame, moved by its modes.
    local_anchors = constraints.anchors + (constraints.modes * bodies.amplitudes[..., constraints.body_idx]).sum(axis=-3)  # [..., constraints, sides] Point
    # A gap is the second anchor's position minus the first's.
    signs = np.array([-1, 1])
    # How each anchor moves for a twist of its body: the commutator with the open twist, in the world frame.
    anchor_motion = (motor >> local_anchors.commutator(Twist)) * signs     # [..., constraints, sides] Direction <- Twist
    # How each anchor moves for each mode amplitude of its body: the mode's shape at the anchor, in the world frame.
    anchor_modes = (motor[..., None, :, :] >> constraints.modes) * signs   # [..., modes, constraints, sides] Direction
    # Each constraint's gap in the world frame, the second anchor's position minus the first's.
    gap = ((motor >> local_anchors) * signs).sum(axis=-1)                  # [..., constraints] Direction
    # Each constraint is a row; each of its two bodies a column.
    constraint_idx = np.arange(constraint_count)[:, None]                  # [constraints, 1]
    # The change in each gap for the bodies' twists.
    rigid = SparseExtensor.from_indices(anchor_motion, constraint_idx, constraints.body_idx, (constraint_count, body_count))  # [..., constraints, bodies] Direction <- Twist
    # The change in each gap for the bodies' amplitudes of each mode, one map per mode.
    modal = SparseExtensor.from_indices(anchor_modes, constraint_idx, constraints.body_idx, (constraint_count, body_count))  # [..., modes] [constraints, bodies] Direction
    # The load on each mode of each body for forces at the constraints, one map per mode.
    modal_load = SparseExtensor.from_indices(Force & anchor_modes, constraints.body_idx, constraint_idx, (body_count, constraint_count))  # [..., modes] [bodies, constraints] Scalar <- Force
    return rigid, modal, modal_load, gap


rigid, modal, modal_load, gap = coupling(bodies, constraints)
print(rigid.gatype, rigid.shape)
print(modal_load.gatype, modal_load.shape)

## 4. Solving all constraints together

`rigid(inverse_inertias(rigid.adjugate()))` maps forces at the constraints to the change in the gaps they cause, `[constraints, constraints] Direction <- Force`. Each mode is a spring with implicit damping. With unit modal mass and its compliance scaled by the step, a mode responds to a unit load by `1 / (1 + compliance)` and adds `modal * (mobility * modal_load)`, summed over the modes, with mobility `compliance / (1 + compliance)`. The constraints' own compliance adds `Force.dual()` times the compliance. One sparse solve of the sum gives the forces at all constraints. Solving the constraints one at a time converges too slowly for a stiff structure to carry load.

In the notation of extended position-based dynamics the system reads as $S = J_b M_b^{-1} J_b^\top + J_q W J_q^\top + \tilde\alpha_p$, with $W = \tilde\alpha_m / (1 + \tilde\alpha_m)$.

In [ ]:
def modal_terms(bodies: Bodies, previous_amplitudes: Scalar, dt: float) -> tuple[Scalar, Scalar, Scalar]:
    """Each mode's compliance, residual and response to a unit load over the step."""
    # The implicit dashpot relative to the spring: twice the damping ratio times the frequency, times the
    # spring's compliance, over the step.
    damping = 2 * bodies.damping * bodies.frequencies * bodies.compliance / dt  # [..., modes, bodies] Scalar
    # The spring's compliance over the step squared, softened by the dashpot.
    compliance = bodies.compliance / dt**2 / (1 + damping)                 # [..., modes, bodies] Scalar
    # The mode's amplitude, with its change over the step weighted in by the dashpot.
    residual = (bodies.amplitudes + damping * (bodies.amplitudes - previous_amplitudes)) / (1 + damping)  # [..., modes, bodies] Scalar
    # A mode has unit mass, so a load on it moves it by 1 / (1 + compliance) of the load.
    return compliance, residual, 1 / (1 + compliance)


def project(bodies: Bodies, constraints: Constraints, previous_amplitudes: Scalar, mode_reactions: Scalar, dt: float) -> tuple[Bodies, Scalar]:
    """The bodies displaced to satisfy all constraints, and the modes' reactions."""
    compliance, residual, response = modal_terms(bodies, previous_amplitudes, dt)  # [..., modes, bodies] Scalar each
    rigid, modal, modal_load, gap = coupling(bodies, constraints)
    # The load of each mode's spring, less the reactions it has already received this step.
    spring_load = -(residual + compliance * mode_reactions)                # [..., modes, bodies] Scalar
    # How far each mode moves under its spring alone.
    unconstrained_step = response * spring_load                            # [..., modes, bodies] Scalar
    # How far each mode moves under a unit load from a constraint, its spring holding back the rest.
    mobility = spdiag(response * compliance)                               # [..., modes] [bodies, bodies] Scalar
    # Each constraint's own compliance over the step squared, as a map from force to gap.
    constraint_compliance = spdiag(constraints.compliance / dt**2 * Force.dual())  # [constraints, constraints] Direction <- Force
    # Each body's inverse inertia, from forque to twist.
    inverse_inertias = spdiag(bodies.inverse_inertia)                      # [..., bodies, bodies] Twist <- Forque
    # The change in the gaps for forces at the constraints: through the bodies' rigid motion, through
    # their modes, summed over the modes, and through the constraints' own compliance.
    system = rigid(inverse_inertias(rigid.adjugate())) + (modal * (mobility * modal_load)).sum(axis=-1) + constraint_compliance  # [..., constraints, constraints] Direction <- Force
    # The reactions at the constraints that close every gap, given how far the modes move on their own.
    # They are position-level, force times the step squared, so through an inverse inertia they give a
    # displacement.
    reactions = system.solve((-gap - (modal * unconstrained_step).sum(axis=-2)).cast(Direction))  # [..., constraints] Force
    # Each body's twist, from the forques the reactions exert on it.
    displacement = bodies.inverse_inertia(rigid.adjugate()(reactions))     # [..., bodies] Twist
    # Each mode's load from the reactions, and its spring's reaction to the whole.
    mode_loads = modal_load(reactions[..., None, :])                       # [..., modes, bodies] Scalar
    reaction_change = response * (spring_load - mode_loads)                # [..., modes, bodies] Scalar
    # The motors moved by the twists, the modes by their loads and the change in their reactions.
    moved = replace(
        bodies,
        motor=bodies.motor * (displacement * -0.5).exp(),
        amplitudes=bodies.amplitudes + mode_loads + reaction_change,
    )
    return moved, mode_reactions + reaction_change

## 5. Time step

Each step integrates the rigid-body equations under gravity with `lie.explicit_rk4` and advances the mode amplitudes by their rates. It then relaxes the modal springs, solves the constraints, and recomputes the rates from the change in position, using the logarithm of the motor's change for the rigid part.

In [ ]:
def step(bodies: Bodies, constraints: Constraints, dt: float, gravity: Direction) -> Bodies:
    """One time step."""
    previous = bodies

    def weight(motor: Motor, rate: Twist) -> Forque:
        # Gravity acts at each body's centre of mass, the origin of its frame.
        return (mv.w.dual() & (motor << gravity)) * bodies.masses            # [..., bodies] Forque

    # Predict the rigid motion from the rigid-body equations under gravity, and move the modes by their rates.
    motor, _ = lie.explicit_rk4(bodies.motor, bodies.rate, bodies.inertia, bodies.inverse_inertia, dt, weight)
    bodies = replace(bodies, motor=motor, amplitudes=bodies.amplitudes + bodies.rates * dt)
    # Relax each mode's spring on its own.
    _, residual, response = modal_terms(bodies, previous.amplitudes, dt)   # [..., modes, bodies] Scalar each
    mode_reactions = -residual * response                                  # [..., modes, bodies] Scalar
    # Displace the bodies so that every constraint holds, the springs relaxing with them.
    bodies, _ = project(replace(bodies, amplitudes=bodies.amplitudes + mode_reactions), constraints, previous.amplitudes, mode_reactions, dt)
    # The rates are the change over the step, the motor's by the logarithm of its change.
    return replace(
        bodies,
        rate=(~previous.motor * bodies.motor).log() * (-2 / dt),
        rates=(bodies.amplitudes - previous.amplitudes) / dt,
    )

## 6. Swinging chain

The chain is released from horizontal under gravity. The motors carry the rotations, and the bending of each girder stays small. Damping acts on the modes only. Inertia is evaluated in the rest configuration and the coupling between rotation and vibration is neglected, which assumes rotation slow compared with the retained vibrations. Contact between girders is not modelled.

In [ ]:
def swing(bodies: Bodies):
    """The bodies' points at every frame, under gravity."""
    for _ in range(FRAMES):
        yield points(bodies, shape)                                          # [cases, bodies, vertices] Point
        for _ in range(SUBSTEPS):
            bodies = step(bodies, constraints, INTERVAL, GRAVITY)


display(Image(filename=save_animation(render.swinging_chain(swing(bodies), shape.edges, points(bodies, shape)), "modal_xpbd_swing", DURATION_MS)))

## 7. Buckling beam

Eight girders with bars of stiffness `1e11`, spliced end to end, are fixed at both ends, and one end is displaced inward by a further step every frame. The beam starts exactly straight, and floating-point rounding is the only perturbation. It stays straight until the end displacement exceeds the critical value, then buckles. A single linear modal body cannot buckle, as its linear stiffness has no geometric stiffness; in a chain of bodies the motors carry the large rotations that provide it. Buckling sets in somewhat after the critical value, since the straight beam remains an unstable equilibrium until the perturbation has grown.

For two chords a height $h$ apart and a span $L$, clamped at both ends, the Euler load $4\pi^2 EI / L^2$ with $EI = EA\,h^2/2$, over the axial stiffness $2EA/L$, gives a critical end displacement of $\pi^2 h^2 / L$.

In [ ]:
beam = core.girder(BEAM_CELLS, float(BEAM_CELLS), BEAM_HEIGHT, BEAM_STIFFNESS, DENSITY, MODES)
clamped, splices = clamped_beam(beam, BEAM_GIRDERS, BEAM_DAMPING)
displacements = END_DISPLACEMENT * np.arange(BEAM_FRAMES) / BEAM_FRAMES      # [frames]


def compress(bodies: Bodies):
    """The beam's points and midspan point at every frame."""
    rest, middle = bodies.motor, bodies.motor.shape[-1] // 2
    for displacement in displacements:
        bodies = displaced(bodies, rest, displacement)
        for _ in range(BEAM_SUBSTEPS):
            bodies = step(bodies, splices, BEAM_INTERVAL, NO_GRAVITY)
        centres = bodies.motor[..., middle - 1:middle + 1] >> mv.w.dual()    # [cases, 2] Point
        yield points(bodies, beam), centres.sum(axis=-1) / 2                 # [cases, bodies, vertices] Point, [cases] Point


frames, midspans = zip(*compress(clamped))
midspans = stack(midspans)                                                   # [frames, cases] Point
euler = critical(BEAM_HEIGHT, BEAM_GIRDERS * BEAM_CELLS)
display(Image(filename=save_animation(render.beam(list(frames), beam.edges), "modal_xpbd_buckle", DURATION_MS)))
render.buckling(displacements, midspans, euler);

In [ ]:
# checks
# The girder and the step laid out here are the core's, the girder to the eigensolver's convergence.
np.testing.assert_allclose(core.girder(CELLS, LENGTH, HEIGHT, STIFFNESS, DENSITY, MODES).frequencies.kernel, shape.frequencies.kernel, rtol=1e-6)
moved, expected = bodies, bodies
for _ in range(SUBSTEPS):
    moved, expected = step(moved, constraints, INTERVAL, GRAVITY), core.step(expected, constraints, INTERVAL, GRAVITY)
np.testing.assert_allclose((points(moved, shape) - core.points(expected, shape)).kernel, 0.0, atol=1e-12)
# rigid.adjugate()(forces) & twists == forces & rigid(twists), summed over the elements.
rigid, _, _, gap = coupling(moved, constraints)
forces = mv.vector(np.random.default_rng(0).normal(size=(1, len(constraints.body_idx), 3))).cast(Force)  # [cases, constraints] Force
twists = mv.bivector(np.random.default_rng(1).normal(size=(1, LINKS + 1, 3)))  # [cases, bodies] Twist
np.testing.assert_allclose((rigid.adjugate()(forces) & twists).sum(axis=-1).kernel, (forces & rigid(twists)).sum(axis=-1).kernel, rtol=1e-12)
# The hinge gaps are below 1e-4.
assert np.abs(gap.kernel).max() < 1e-4
# The beam is straight below 0.8 of the critical displacement and buckled above twice it.
deflection = (midspans[:, 0].dual() | mv.y).abs()                            # [frames] Scalar
assert deflection[displacements < 0.8 * euler].kernel.max() < 1e-6
assert deflection[displacements > 2 * euler].kernel.min() > 0.5